# 09 — Explainability (Colab + Drive)

Needs 01 + **trained notebook 07** (`checkpoints/stage2/best_attention.pt`). GPU Off.

Loads the Stage 2 model and writes **real** per-concept attention — not placeholders.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q matplotlib tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
MEL_CACHE = Path("/content/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


def load_mel_npy(mel_abs, retries=5, pause=2.0):
    """Load mel from Drive with retries; cache on Colab disk to avoid FUSE drops."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-download its shard in notebook 00: {mel_abs} "
        f"({nbytes} bytes on Drive). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Guard + real Stage 2 attention


In [ ]:
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset

if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first")

CKPT_PATH = CKPT_DIR / "stage2" / "best_attention.pt"
if not CKPT_PATH.exists() or CKPT_PATH.stat().st_size == 0:
    raise FileNotFoundError(
        "Notebook 09 refuses placeholder attention. Train notebook 07 first so this exists and is non-empty:\n"
        f"  {CKPT_PATH}\n"
        "07 must be trained AFTER notebook 04 writes AcousticBrainz rhythm features."
    )

rhythm_csv = FEAT_DIR / "rhythm" / "rhythm_song.csv"
if not rhythm_csv.exists():
    raise FileNotFoundError("Run notebook 04 first (AcousticBrainz rhythm).")
rhythm_src = pd.read_csv(rhythm_csv)
if "source" in rhythm_src.columns and (rhythm_src["source"] == "mel_proxy").any():
    raise RuntimeError("rhythm_song.csv still has mel_proxy rows — re-run notebook 04, then re-train 07.")

manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
E = np.load(FEAT_DIR / "instrument" / "instrument_embeddings.npy")
inst_ids = json.loads((FEAT_DIR / "instrument" / "song_ids.json").read_text())
inst_map = {normalize_track_id(s) or str(s): E[i] for i, s in enumerate(inst_ids)}


def load_feat(sub):
    p = FEAT_DIR / sub / f"{sub}_song.csv"
    if not p.exists():
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    df["song_id"] = df["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
    return df.set_index("song_id")


rhythm, timbre, harmony = load_feat("rhythm"), load_feat("timbre"), load_feat("harmony")

def ncols(df):
    return [c for c in df.columns if c not in ("source", "split") and pd.api.types.is_numeric_dtype(df[c])]

r_cols, t_cols, h_cols = ncols(rhythm), ncols(timbre), ncols(harmony)
have = set(inst_map) & set(rhythm.index) & set(timbre.index) & set(harmony.index)
manifest = manifest[manifest["song_id"].isin(have)].copy()
test_ids = manifest.loc[manifest.split == "test", "song_id"].astype(str).head(12).tolist()
if not test_ids:
    raise RuntimeError("No overlapping test songs with all four concepts — check 03–06 then 07.")
print("test sample", test_ids)
print("using checkpoint", CKPT_PATH, "bytes=", CKPT_PATH.stat().st_size)

ids = manifest["song_id"].astype(str).tolist()
id_to_idx = {s: i for i, s in enumerate(ids)}
tag_to_idx, rows = {}, {s: set() for s in ids}
for path in [ANN_DIR / "autotagging_genre.tsv", *ANN_DIR.rglob("*genre*.tsv")]:
    if not Path(path).exists():
        continue
    for rec in iter_tsv_rows(Path(path)):
        sid = normalize_track_id(rec["TRACK_ID"])
        if sid not in rows:
            continue
        for tag in rec.get("TAGS", "").replace("|", "\t").split("\t"):
            leaf = tag.strip().split("/")[-1].split("---")[-1]
            if leaf and leaf.lower() not in {"nan", "tags", ""}:
                tag_to_idx.setdefault(leaf, len(tag_to_idx))
                rows[sid].add(leaf)
    if tag_to_idx:
        break
n_tags = max(len(tag_to_idx), 1)
Y = np.zeros((len(ids), n_tags), np.float32)
for i, sid in enumerate(ids):
    for t in rows[sid]:
        Y[i, tag_to_idx[t]] = 1.0


class DS(Dataset):
    def __init__(self, song_ids):
        self.ids = list(song_ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        sid = self.ids[i]
        inst = inst_map[sid].astype(np.float32)
        r = rhythm.loc[sid, r_cols].astype(np.float32).fillna(0).values
        t = timbre.loc[sid, t_cols].astype(np.float32).fillna(0).values
        h = harmony.loc[sid, h_cols].astype(np.float32).fillna(0).values
        return torch.tensor(inst), torch.tensor(r), torch.tensor(t), torch.tensor(h), sid


class AttentionFusion(nn.Module):
    def __init__(self, d_i, d_r, d_t, d_h, token=64, fused=128, n_tags=87):
        super().__init__()
        self.p_i, self.p_r, self.p_t, self.p_h = (
            nn.Linear(d_i, token), nn.Linear(d_r, token), nn.Linear(d_t, token), nn.Linear(d_h, token)
        )
        self.attn = nn.MultiheadAttention(token, 1, batch_first=True)
        self.out = nn.Sequential(nn.Linear(token, fused), nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Linear(fused, n_tags)

    def forward(self, inst, r, t, h):
        tok = torch.stack([self.p_i(inst), self.p_r(r), self.p_t(t), self.p_h(h)], 1)
        o, w = self.attn(tok, tok, tok, need_weights=True)
        return self.head(self.out(o.mean(1))), w


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
n_tags = len(state.get("tags") or []) or n_tags
model = AttentionFusion(64, len(r_cols), len(t_cols), len(h_cols), n_tags=n_tags).to(DEVICE)
model.load_state_dict(state["model"])
model.eval()

concept_names = ["instrument", "rhythm", "timbre", "harmony"]
attn_rows = []
with torch.no_grad():
    for inst, r, t, h, sid in DataLoader(DS(test_ids), batch_size=8, num_workers=0):
        _, w = model(inst.to(DEVICE), r.to(DEVICE), t.to(DEVICE), h.to(DEVICE))
        # w: (B, 4, 4) — mean over query tokens → per-concept weight
        weights = w.mean(dim=1).cpu().numpy()
        for i, song in enumerate(sid):
            row = {"song_id": str(song)}
            row.update({c: float(weights[i, j]) for j, c in enumerate(concept_names)})
            attn_rows.append(row)

attn_df = pd.DataFrame(attn_rows)
attn_df.to_csv(RESULTS_DIR / "09_attention_weights_sample.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(concept_names, attn_df[concept_names].mean(0).values)
ax.set_title("Mean concept attention (Stage 2, test sample)")
ax.set_ylabel("mean attention")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "09_mean_attention.png", dpi=150)
plt.show()
qual = []
for sid in test_ids[:5]:
    top = concept_names[int(attn_df.loc[attn_df.song_id == sid, concept_names].values.argmax())]
    qual.append({"song_id": sid, "audible_dominant_concept": "", "model_top_concept": top, "agree": "", "comment": ""})
pd.DataFrame(qual).to_csv(RESULTS_DIR / "09_qualitative_listening.csv", index=False)
print("Wrote real attention under", RESULTS_DIR)
attn_df
